In [1]:
import sys
from pathlib import Path
import pandas as pd

projectRoot = Path().resolve().parents[0]
print(f"path: {projectRoot}")
sys.path.append(str(projectRoot))

%load_ext autoreload
%autoreload 2

path: /workspace


In [2]:
# Extract canonical facts from FCA HTML files
from src.parsing.arelle_parser import process_html_files

html_folder = projectRoot / "data" / "raw" / "fca"
output_path = (projectRoot / "data" / "processed" / 
               "canonical_facts" / "bronze.csv")

if output_path.exists():
    print(f"Bronze data already exists at: {output_path}")
    bronze_df = pd.read_csv(output_path)
else:
    bronze_df = process_html_files(html_folder, output_path, debug=False)

sample_bronze = bronze_df.sample(n=250, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "sample_bronze.csv")
sample_bronze.to_csv(csv_path, index=False)
print(f"Bronze sample extract saved to: {csv_path}")

Bronze data already exists at: /workspace/data/processed/canonical_facts/bronze.csv
Bronze sample extract saved to: /workspace/data/processed/debug/sample_bronze.csv


In [3]:
# Create silver ground truth from bronze data
from src.parsing.canonical_facts import create_silver_ground_truth

silver_df = create_silver_ground_truth(bronze_df, 
                        str(projectRoot / "data" / "processed" /
                            "canonical_facts" / "silver.csv"))

sample_silver = silver_df.sample(n=300, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "sample_silver.csv")
sample_silver.to_csv(csv_path, index=False)
print(f"Silver sample extract saved to: {csv_path}")

Created silver ground truth with 7291 rows.
Silver data saved to: /workspace/data/processed/canonical_facts/silver.csv
Silver sample extract saved to: /workspace/data/processed/debug/sample_silver.csv


In [4]:
# Create gold ground truth from silver data
from src.parsing.canonical_facts import create_gold_ground_truth

gold_df = create_gold_ground_truth(silver_df, 
                         str(projectRoot / "data" / "processed" /
                              "canonical_facts" / "gold.csv"))

sample_gold = gold_df.sample(n=300, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "sample_gold.csv")
sample_gold.to_csv(csv_path, index=False)
print(f"Gold sample extract saved to: {csv_path}")

Created gold ground truth with 3144 rows.
Gold data saved to: /workspace/data/processed/canonical_facts/gold.csv
Gold sample extract saved to: /workspace/data/processed/debug/sample_gold.csv


In [5]:
# Generate TINY gold sample
sample_gold_tiny = gold_df[~gold_df['segment'].isin(['Narrative_Disclosure', 
                        'Other_Financial_Metric'])].sample(n=10, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "tiny_sample_gold.csv")
sample_gold_tiny.to_csv(csv_path, index=False)
print(f"Tiny Gold sample extract saved to: {csv_path}")

Tiny Gold sample extract saved to: /workspace/data/processed/debug/tiny_sample_gold.csv


In [ ]:
# Split qa_pairs.csv into smaller chunks to stay within the 90,000 enqueued tokens limit
import pandas as pd

# Load the QA pairs
qa_pairs_file = "/workspace/data/qa/llm_batch/qa_pairs.csv"
qa_pairs = pd.read_csv(qa_pairs_file)

# Define the maximum number of rows per chunk
max_rows_per_chunk = 250  # Adjust this value based on your token estimation

# Split the dataframe into chunks
chunk_dir = "/workspace/data/qa/llm_batch/chunks"
Path(chunk_dir).mkdir(parents=True, exist_ok=True)

for i, chunk in enumerate(range(0, len(qa_pairs), max_rows_per_chunk)):
    chunk_file = f"{chunk_dir}/qa_pairs_chunk_{i + 1}.csv"
    qa_pairs.iloc[chunk:chunk + max_rows_per_chunk].to_csv(chunk_file, index=False)
    print(f"Chunk saved to: {chunk_file}")